# Router / Oracle / AllClasses — baseline full run 03b9b53
Evaluation only, no training or router refitting. Attach the original 100-clients Kaggle dataset. The notebook searches `/kaggle/input` for the folder containing both `metadata.json` and `global_test_data.npz` (and prefers the historical path if present). It downloads `baseline full run 03b9b53.zip` from Google Drive. Enable Internet and GPU, paste its share URL in the first code cell, then Run All.

Uses task 5 round 19, 50k class-balanced samples, identical client shards across policies. Oracle uses ground-truth task from dataset metadata and that task's adapter plus global task class mask. Router uses predicted adapter and local router mask. AllClasses disables adapters and predicts over all globally seen classes. Oracle is a task-informed diagnostic, not deployable or a guaranteed upper bound when adapter paths differ. Conditional correct-route accuracy is not Oracle accuracy.

This is the selected baseline checkpoint before the AMP/peer fixes (training commit 03b9b53). Delta storage can cause small reconstruction differences from the original in-memory score. Report pooled and client-mean metrics separately. No forgetting claim can be made from this single checkpoint.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile, shutil, gc
RESULTS_DRIVE_URL = "https://drive.google.com/file/d/1BEjP4iGJbPT0uFcHZ7WXX4oM_1vyvx0M/view?usp=sharing"
RESULTS_ZIP = Path('/kaggle/working/results_03b9b53_full.zip')
DATA_DIR = "" # auto-discover mounted original dataset
TASK, ROUND, MAX_SAMPLES, BATCH_SIZE = 5, 19, 50000, 1024
OUT = Path('/kaggle/working/denice_oracle_03b9b53')
OUT.mkdir(parents=True, exist_ok=True)
SOURCE = Path('/kaggle/working/denice_oracle_source')
if not SOURCE.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/khoilv2005/FL_IL_IDS.git',str(SOURCE)],
                   check=True, env=dict(os.environ,GIT_LFS_SKIP_SMUDGE='1'))
source_commit = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()
sys.path.insert(0,str(SOURCE))
print('Evaluation source:', source_commit)


In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from fed_learning.data.incremental_loader import IncrementalDataLoader
from fed_learning.training.denice_delta_checkpoint import load_denice_checkpoint
from fed_learning.training.decentralized_denice_il import _limit_eval_samples, _partition_test_data_by_client
from fed_learning.training.denice_eval import _denice_routed_logits_with_episodes
from eval_checkpoint import _make_denice_client_model
if not DATA_DIR or not (Path(DATA_DIR, 'metadata.json').exists() and Path(DATA_DIR, 'global_test_data.npz').exists()):
    input_root = Path('/kaggle/input')
    candidates = sorted({p.parent for p in input_root.rglob('metadata.json')
                         if (p.parent/'global_test_data.npz').exists()})
    historical = Path('/kaggle/input/datasets/khoilv2005/100-clients/100-clients')
    if historical in candidates:
        DATA_DIR = str(historical)
    elif len(candidates) == 1:
        DATA_DIR = str(candidates[0])
    else:
        raise FileNotFoundError(f'Could not uniquely find original dataset under {input_root}. Candidates: {candidates}. Attach the 100-clients dataset or set DATA_DIR.')
print('Using dataset:', DATA_DIR)
if 'PASTE_GOOGLE_DRIVE' in RESULTS_DRIVE_URL:
    raise ValueError('Paste the Google Drive share URL for the full-run ZIP in RESULTS_DRIVE_URL.')
if not RESULTS_ZIP.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    subprocess.run([sys.executable, '-m', 'gdown', '--fuzzy', RESULTS_DRIVE_URL, '-O', str(RESULTS_ZIP)], check=True)
if not RESULTS_ZIP.exists() or RESULTS_ZIP.stat().st_size == 0:
    raise FileNotFoundError(f'Google Drive download failed: {RESULTS_ZIP}')
name = f'checkpoint_task_{TASK}_round_{ROUND}.pt'
root = OUT/'checkpoints'
root.mkdir(exist_ok=True)
with zipfile.ZipFile(RESULTS_ZIP) as z:
    matches = [n for n in z.namelist() if Path(n).name == name]
    if len(matches) != 1: raise ValueError(f'Expected one {name}; found {len(matches)}')
    prefix = matches[0][:-len(name)]
    for n in z.namelist():
        if not n.startswith(prefix) or n.endswith('/'): continue
        base = Path(n).name
        if base in ('config.json','final_report.json','checkpoint_index.json') or base == f'checkpoint_task_{TASK}_base.pt' or base.startswith(f'checkpoint_task_{TASK}_round_'):
            with z.open(n) as src, (root/base).open('wb') as dst: shutil.copyfileobj(src,dst)
config = json.loads((root/'config.json').read_text())
report = json.loads((root/'final_report.json').read_text())
if not str(config.get('git_commit','')).startswith('03b9b53'):
    raise ValueError('Expected baseline full run, training commit 03b9b53')
if report['completed_task'] != TASK: raise ValueError('Report must match final task')
ckpt = load_denice_checkpoint(str(root/name))
ids = [int(cid) for cid in report['final_metrics']['per_client']]
if not ids or not set(ids) <= {int(cid) for cid in ckpt['client_model_states']}:
    raise ValueError('Missing original evaluation clients')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Training source:',config['git_commit'],'clients:',len(ids),'device:',device)


In [ ]:
loader = IncrementalDataLoader(DATA_DIR)
classes = {t:list(map(int,loader.get_task_classes(t))) for t in range(TASK+1)}
mapping = {}
for t, labels in classes.items():
    for label in labels:
        if label in mapping: raise ValueError('Ambiguous class-to-task map')
        mapping[label]=t
seen = sorted(mapping)
seed = int(config.get('random_seed',config.get('seed',42)))
x,y = loader.get_test_data(TASK,cumulative=True)
if not len(y): raise ValueError('Empty test data')
x,y,sample_info = _limit_eval_samples(x,y,MAX_SAMPLES,seed+TASK)
shards,partition = _partition_test_data_by_client(x,y,ids,seed+104729*TASK)
loader._test_data=None
del x,y
gc.collect()
protocol = dict(training_commit=config['git_commit'],evaluation_commit=source_commit,
    checkpoint=name,storage='fp16 delta reconstruction',results_path=str(RESULTS_ZIP),
    data_dir=DATA_DIR,task_classes=classes,client_ids=ids,sample_info=sample_info,partition=partition,
    policies={'Router':'predicted adapter + local task mask',
              'Oracle':'true task adapter + global dataset task mask',
              'AllClasses':'no adapter + globally seen classes'})
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2))
print(sample_info)


In [ ]:
policies = ['Router','Oracle','AllClasses']
@torch.no_grad()
def predict(model,detector,xb,tasks,policy):
    if policy=='Router':
        logits,_ = _denice_routed_logits_with_episodes(model,xb,detector,seen,device,inference_policy='pred_hard')
        return logits.argmax(1)
    model.clear_active_adapters()
    if policy=='AllClasses':
        logits=model(xb).float()
        mask=torch.zeros(logits.shape[1],dtype=torch.bool,device=device); mask[seen]=True
        return logits.masked_fill(~mask[None,:],-torch.inf).argmax(1)
    pred=torch.empty(len(xb),dtype=torch.long,device=device)
    try:
        for task in np.unique(tasks):
            idx=torch.as_tensor(np.flatnonzero(tasks==task),device=device)
            model.clear_active_adapters(); model.set_active_context(int(task))
            logits=model(xb.index_select(0,idx)).float()
            mask=torch.zeros(logits.shape[1],dtype=torch.bool,device=device); mask[classes[int(task)]]=True
            pred[idx]=logits.masked_fill(~mask[None,:],-torch.inf).argmax(1)
    finally:
        model.clear_active_adapters()
    return pred

rows,frames = [],[]
for pos,cid in enumerate(ids,1):
    model,detector=_make_denice_client_model(ckpt,cid,device)
    model.eval()
    stored=ckpt['client_model_states'].get(cid,ckpt['client_model_states'].get(str(cid)))
    if set(stored)!=set(model.state_dict()): raise ValueError(f'Client {cid}: checkpoint key mismatch')
    sx,sy=shards[cid]['X_test'],shards[cid]['y_test']
    truth=sy.numpy()
    tasks=np.asarray([mapping[int(v)] for v in truth])
    known={int(c) for labels in detector.episode_classes.values() for c in labels}
    covered=np.isin(truth,list(known))
    predictions={}
    for policy in policies:
        pieces=[]
        for start in range(0,len(sy),BATCH_SIZE):
            end=start+BATCH_SIZE
            pieces.append(predict(model,detector,sx[start:end].to(device),tasks[start:end],policy).cpu().numpy())
        pred=np.concatenate(pieces); predictions[policy]=pred
        rows.append(dict(client_id=cid,policy=policy,samples=len(truth),
            accuracy=float(accuracy_score(truth,pred)),
            f1_macro=float(f1_score(truth,pred,average='macro',zero_division=0)),
            coverage=float(covered.mean()),
            covered_accuracy=float((pred[covered]==truth[covered]).mean()) if covered.any() else None))
    frames.append(pd.DataFrame(dict(client_id=cid,sample_in_shard=np.arange(len(truth)),
                  y_true=truth,true_task=tasks,covered=covered,**predictions)))
    pd.DataFrame(rows).to_csv(OUT/'per_client_metrics.csv',index=False)
    print(f'Client {pos}/{len(ids)} cid={cid}:',
          {p:round(float((predictions[p]==truth).mean())*100,2) for p in policies},flush=True)
    del model,detector
    if device=='cuda': torch.cuda.empty_cache()
preds=pd.concat(frames,ignore_index=True)
preds.to_csv(OUT/'predictions.csv',index=False)
summary=[]
for p in policies:
    items=[r for r in rows if r['policy']==p]
    summary.append(dict(policy=p,accuracy=float(accuracy_score(preds.y_true,preds[p])),
        f1_macro=float(f1_score(preds.y_true,preds[p],labels=seen,average='macro',zero_division=0)),
        client_mean_accuracy=float(np.mean([r['accuracy'] for r in items])),
        client_mean_f1_macro=float(np.mean([r['f1_macro'] for r in items]))))
summary=pd.DataFrame(summary).set_index('policy')
summary['delta_vs_router_pp']=100*(summary.accuracy-summary.loc['Router','accuracy'])
summary.to_csv(OUT/'summary.csv')
display(summary)
print('Original client-mean accuracy:',report['final_metrics']['accuracy'])
print('Restored Router client-mean accuracy:',summary.loc['Router','client_mean_accuracy'])
print('Fractions for accuracy/F1; percentage points for delta. Reconstruction/protocol drift must not be attributed to Oracle.')


In [ ]:
task_rows=[]
for task,group in preds.groupby('true_task'):
    for policy in policies:
        task_rows.append(dict(task=int(task),policy=policy,samples=len(group),
            accuracy=float(accuracy_score(group.y_true,group[policy])),
            f1_macro=float(f1_score(group.y_true,group[policy],labels=classes[int(task)],average='macro',zero_division=0))))
table=pd.DataFrame(task_rows)
table.to_csv(OUT/'per_task_summary.csv',index=False)
pivot=table.pivot(index='task',columns='policy',values='accuracy')[policies]
display(pivot)
import matplotlib.pyplot as plt
ax=pivot.plot.bar(figsize=(10,4))
ax.set_ylim(0,1); ax.set_ylabel('Accuracy')
ax.set_title('Frozen task-5 checkpoint: accuracy by test task')
plt.tight_layout(); plt.savefig(OUT/'accuracy_by_task.png',dpi=180); plt.show()
with zipfile.ZipFile('/kaggle/working/denice_oracle_diagnostics.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file(): z.write(p,p.name)
print('Download /kaggle/working/denice_oracle_diagnostics.zip')
